In [ ]:
# Uses Python's built-in ast module; no package installation is required.


## Load actual scanner output and source files


In [ ]:
from pathlib import Path

SCAN_RESULTS_PATH = Path(input("Path to the REAL scanner results JSON: ").strip().strip('"')).expanduser().resolve()
SOURCE_ROOT = Path(input("Path to the directory containing the scanned source: ").strip().strip('"')).expanduser().resolve()
ENTRY_POINTS = [
    name.strip()
    for name in input("Configured Python entry-point function names (comma-separated): ").split(",")
    if name.strip()
]

if not SCAN_RESULTS_PATH.is_file():
    raise FileNotFoundError(f"Scanner results JSON not found: {SCAN_RESULTS_PATH}")
if not SOURCE_ROOT.is_dir():
    raise NotADirectoryError(f"Source root directory not found: {SOURCE_ROOT}")
if not ENTRY_POINTS:
    raise ValueError("Enter at least one configured entry-point function name.")

print(f"Using scanner output: {SCAN_RESULTS_PATH}")
print(f"Matching source under: {SOURCE_ROOT}")
print(f"Configured entry points: {', '.join(ENTRY_POINTS)}")


## Conservative, per-file Python AST reachability


In [ ]:
import sys
from pathlib import Path

ANALYZER_DIR = Path.cwd()
if not (ANALYZER_DIR / "noise_filter.py").is_file():
    ANALYZER_DIR = ANALYZER_DIR / "3_ast_analyzer"
if not (ANALYZER_DIR / "noise_filter.py").is_file():
    raise FileNotFoundError("Could not locate 3_ast_analyzer/noise_filter.py")
if str(ANALYZER_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYZER_DIR))

from noise_filter import ASTNoiseFilter

print("Conservative Python AST reachability analyzer loaded.")


## Live Output Execution

In [ ]:
filter_engine = ASTNoiseFilter(main_entry_points=ENTRY_POINTS)
reachable_findings, review_findings = filter_engine.process_scan_file(
    SCAN_RESULTS_PATH, SOURCE_ROOT
)

print("=" * 65)
print("  AST REACHABILITY RESULTS FOR THE PROVIDED SCAN")
print("=" * 65)
print(f"Scanner findings                 : {len(reachable_findings) + len(review_findings)}")
print(f"Statically reachable             : {len(reachable_findings)}")
print(f"Kept for review (uncertain)      : {len(review_findings)}")
print("Reachability does not validate whether a finding is a vulnerability.")
print("No findings are discarded as dead code.")

for label, findings in (("STATICALLY REACHABLE", reachable_findings), ("KEEP FOR REVIEW", review_findings)):
    print(f"\n[{label}]")
    for item in findings:
        location = item.get("file") or item.get("path") or item.get("file_path", "unknown file")
        line = item.get("line") or item.get("line_number", "?")
        function = item.get("function") or item.get("function_name", "not mapped")
        print(f"- {item.get('rule_id', 'unknown rule')}: {function}")
        print(f"  {location} (line {line}) - {item['status']}")
        evidence = item.get("reachability_evidence", {})
        print(f"  Evidence: {evidence.get('reason', 'No static evidence available.')}")
        if evidence.get("call_path"):
            print(f"  Call path: {' -> '.join(evidence['call_path'])}")


## Run standalone with the same real inputs


In [ ]:
print("Use noise_filter.py with these actual inputs:")
print(f'  --scan "{SCAN_RESULTS_PATH}"')
print(f'  --source-root "{SOURCE_ROOT}"')
for entry_point in ENTRY_POINTS:
    print(f'  --entry-point "{entry_point}"')
print("The script requires all input paths and entry points; it has no sample-data default.")
